In [1]:
import numpy as np

In [2]:
class LinearRegression:
    def __init__(self, solving_method='analit', learning_rate=0.01, max_iterations=1000, epsilon=1e-8):
        self.solving_method = solving_method
        self.learning_rate = learning_rate
        self.max_iterations = max_iterations
        self.epsilon = epsilon
        self.feature_weights = None
        self.intercept = None
        self.all_weights = None
        self.num_features = None
        self.is_trained = False

    def fit(self, X, y):
        X = np.asarray(X)
        if X.ndim == 0 or X.ndim > 2:
            raise ValueError

        if X.ndim == 1:
            X = X.reshape(-1, 1)

        y = np.asarray(y)

        if y.ndim == 0 or y.ndim > 2:
            raise ValueError

        if y.ndim == 2:
            if y.shape[0] > 1 and y.shape[1] > 1:
                raise ValueError
            y = y.reshape(-1)

        if X.shape[0] != y.shape[0]:
            raise ValueError
        d = X.shape[1]

        X_b = np.insert(X, 0, 1, axis=1)

        if self.solving_method == 'analit':
            w = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
            #w = np.linalg.lstsq(X_b, y, rcond=None)[0]

        elif self.solving_method == 'gd':
            theta = np.random.randn(d + 1)
            for i in range(self.max_iterations):
                y_pred = X_b @ theta
                grad = X_b.T @ (-1 * (y - y_pred)) / X_b.shape[0]
                new_theta = theta - self.learning_rate * grad
                if np.linalg.norm(new_theta - theta) < self.epsilon:
                    theta = new_theta #############################################################
                    break
                else:
                    theta = new_theta
            w = theta

            self.intercept = float(w[0])
            self.feature_weights = w[1:]
            self.num_features = d
            self.is_trained = True
            self.all_weights = w

            return self

        else:
            raise NotImplementedError

        self.intercept = float(w[0])
        self.feature_weights = w[1:]
        self.num_features = d
        self.is_trained = True
        self.all_weights = w

        return self

    def predict(self, X):
        if self.is_trained == False:
            raise RuntimeError

        X = np.asarray(X)
        if X.ndim == 0 or X.ndim > 2:
            raise ValueError
        if X.ndim == 1:
            X = X.reshape(-1, 1)

        if X.shape[1] != self.num_features:
            raise ValueError

        X_b = np.insert(X, 0, 1, axis=1)

        y = X_b @ self.all_weights
        return y

    def score(self, X, y):
        if self.is_trained == False:
            raise RuntimeError

        y_pred = self.predict(X)

        y = np.asarray(y)
        if y.ndim == 0 or y.ndim > 2:
            raise ValueError
        if y.ndim == 2:
            if y.shape[0] > 1 and y.shape[1] > 1:
                raise ValueError
            y = y.reshape(-1)

        if y.shape[0] != y_pred.shape[0]:
            raise ValueError

        diff = np.subtract(y, y_pred)
        diff_abs = np.abs(diff)
        diff_square = np.square(diff)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        mae = float(np.mean(diff_abs))
        mse = float(np.mean(diff_square))

        if ss_tot == 0:
            if ss_res == 0:
                r_2 = 1.0
            else:
                r_2 = 0.0
        else:
            r_2 = float(1 - (ss_res / ss_tot))

        return mae, mse, r_2

In [4]:
x = np.array([5, 15, 25, 35, 45, 55]).reshape((-1, 1))
y = np.array([5, 20, 14, 32, 22, 38])

print("x =", x.ravel())
print("y =", y)
print()

model_ols = LinearRegression(solving_method='analit')
model_ols.fit(x, y)

print("intercept (ols) =", model_ols.intercept)
print("feature_weights (ols) =", model_ols.feature_weights)
print("all_weights (ols) =", model_ols.all_weights)
print("num_features =", model_ols.num_features)
print("is_trained =", model_ols.is_trained)
print()

y_pred_ols = model_ols.predict(x)
print("y_pred (ols)           =", y_pred_ols)

mae_ols, mse_ols, r2_ols = model_ols.score(x, y)
print("mae (ols) =", mae_ols)
print("mse (ols) =", mse_ols)
print("r2  (ols) =", r2_ols)
print()

model_gd = LinearRegression(
    solving_method='gd',
    learning_rate=0.001,
    max_iterations=100000,
    epsilon=1e-10,
)
model_gd.fit(x, y)

print("intercept (gd) =", model_gd.intercept)
print("feature_weights (gd) =", model_gd.feature_weights)
print("all_weights (gd) =", model_gd.all_weights)
print()

y_pred_gd = model_gd.predict(x)
print("y_pred (gd) =", y_pred_gd)

mae_gd, mse_gd, r2_gd = model_gd.score(x, y)
print("mae (gd) =", mae_gd)
print("mse (gd) =", mse_gd)
print("r2  (gd) =", r2_gd)
print()

print("all_weights_diff (gd - ols) =", model_gd.all_weights - model_ols.all_weights)

x = [ 5 15 25 35 45 55]
y = [ 5 20 14 32 22 38]

intercept (ols) = 5.633333333333339
feature_weights (ols) = [0.54]
all_weights (ols) = [5.63333333 0.54      ]
num_features = 1
is_trained = True

y_pred (ols)           = [ 8.33333333 13.73333333 19.13333333 24.53333333 29.93333333 35.33333333]
mae (ols) = 5.466666666666666
mse (ols) = 33.755555555555546
r2  (ols) = 0.7158756137479543

intercept (gd) = 5.633332924791325
feature_weights (gd) = [0.54000001]
all_weights (gd) = [5.63333292 0.54000001]

y_pred (gd) = [ 8.33333298 13.73333308 19.13333318 24.53333328 29.93333339 35.33333349]
mae (gd) = 5.46666661523124
mse (gd) = 33.755555555555596
r2  (gd) = 0.7158756137479538

all_weights_diff (gd - ols) = [-4.08542014e-07  1.02870852e-08]
